In [4]:
!git clone https://github.com/meta-compute/sem-fracture-deep-learning.git

Cloning into 'sem-fracture-deep-learning'...
remote: Enumerating objects: 2691, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 2691 (delta 0), reused 3 (delta 0), pack-reused 2685 (from 2)
Receiving objects: 100% (2691/2691), 904.02 MiB | 40.42 MiB/s, done.
Resolving deltas: 100% (17/17), done.
Updating files: 100% (2000/2000), done.
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [10]:
cd sem-fracture-deep-learning

/kaggle/working/sem-fracture-deep-learning


In [12]:
!git checkout vishnu-dev
!pip install -r requirements.txt

Updating files: 100% (635/635), done.
branch 'vishnu-dev' set up to track 'origin/vishnu-dev'.
Switched to a new branch 'vishnu-dev'


# Fracture modelling

Use the sections below in order: Train, Validate, then optionally Test. Mixed augmentation is training-only. Final test evaluation is disabled until explicitly enabled. The stored training output is from an earlier interrupted attempt.


## Train

Configure the model in the setup cell and run training. The best validation-macro-F1 checkpoint is saved under `results/shared_pipeline/`.


In [13]:
from pathlib import Path

import pandas as pd
import torch

from sem_fracture.pipeline import run_experiment, run_prediction

DATA_DIR = Path.cwd()
MODEL_KEY = "resnet18"  # resnet18, deit_tiny, tinyvit_11m, tinyvit_21m
MODE = "finetune"  # head_only or finetune
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

BASELINE_CHECKPOINTS = {
    "resnet18": DATA_DIR / "results" / "resnet18_best.pth",
    "deit_tiny": DATA_DIR / "results" / "deit_tiny" / "checkpoints" / "deit_tiny_best.pth",
    "tinyvit_11m": DATA_DIR / "results" / "tinyvit_11m" / "checkpoints" / "tinyvit_11m_best.pth",
    "tinyvit_21m": DATA_DIR / "results" / "tinyvit_21m" / "checkpoints" / "tinyvit_21m_best.pth",
}

if MODE not in ("head_only", "finetune"):
    raise ValueError(f"Unknown training mode: {MODE}")
if MODEL_KEY not in BASELINE_CHECKPOINTS:
    raise ValueError(f"Unknown model key: {MODEL_KEY}")

INITIAL_CHECKPOINT = BASELINE_CHECKPOINTS[MODEL_KEY] if MODE == "finetune" else None
BACKBONE_LR = 1e-5 if MODE == "finetune" else None
HEAD_LR = 1e-4 if MODE == "finetune" else 1e-3
EPOCHS = 30 if MODEL_KEY == "tinyvit_21m" and MODE == "finetune" else 10
PATIENCE = 4 if MODE == "finetune" else None
GRAD_CLIP_NORM = 1.0 if MODEL_KEY.startswith("tinyvit") and MODE == "finetune" else None
CHECKPOINT_PATH = (
    DATA_DIR / "results" / "shared_pipeline"
    / f"{MODEL_KEY}_{MODE}_best.pth"
)
RUN_FINAL_TEST_EVALUATION = False  # enable only after selecting the checkpoint


In [14]:
result = run_experiment(
    data_dir=DATA_DIR,
    model_key=MODEL_KEY,
    mode=MODE,
    checkpoint_path=CHECKPOINT_PATH,
    device=DEVICE,
    head_lr=HEAD_LR,
    backbone_lr=BACKBONE_LR,
    weight_decay=1e-4,
    epochs=EPOCHS,
    patience=PATIENCE,
    grad_clip_norm=GRAD_CLIP_NORM,
    initial_checkpoint=INITIAL_CHECKPOINT,
    augment_mixed=True,
    seed=42,
    batch_size=16,
)

best = result["best_checkpoint"]
print("Best epoch:", best["epoch"])
print("Best validation macro-F1:", f"{best['val_macro_f1']:.4f}")
print("Checkpoint:", CHECKPOINT_PATH)
print("Training split:", result["split_file"])
print("Class weights:", result["class_weights"])


Epoch 1/10 starting
Epoch 1/10 | train_loss=0.2043 | val_loss=0.1569 | val_accuracy=0.9500 | val_macro_f1=0.9026 | elapsed=25.6s | best_checkpoint=True
Epoch 2/10 starting
Epoch 2/10 | train_loss=0.1516 | val_loss=0.1426 | val_accuracy=0.9500 | val_macro_f1=0.8991 | elapsed=20.3s | best_checkpoint=False
Epoch 3/10 starting
Epoch 3/10 | train_loss=0.1283 | val_loss=0.1252 | val_accuracy=0.9643 | val_macro_f1=0.9268 | elapsed=20.2s | best_checkpoint=True
Epoch 4/10 starting
Epoch 4/10 | train_loss=0.1074 | val_loss=0.1121 | val_accuracy=0.9679 | val_macro_f1=0.9318 | elapsed=20.2s | best_checkpoint=True
Epoch 5/10 starting
Epoch 5/10 | train_loss=0.1011 | val_loss=0.1245 | val_accuracy=0.9571 | val_macro_f1=0.9124 | elapsed=20.1s | best_checkpoint=False
Epoch 6/10 starting
Epoch 6/10 | train_loss=0.0810 | val_loss=0.1356 | val_accuracy=0.9607 | val_macro_f1=0.9195 | elapsed=20.2s | best_checkpoint=False
Epoch 7/10 starting
Epoch 7/10 | train_loss=0.0757 | val_loss=0.1116 | val_accuracy=0

## Validate

Evaluate the selected checkpoint on the original validation split. Validation data is not augmented.


In [15]:
validation_result = run_prediction(
    data_dir=DATA_DIR,
    model_key=MODEL_KEY,
    checkpoint_path=CHECKPOINT_PATH,
    split="validation",
    split_file="dataset_split_augmented.csv",
    device=DEVICE,
    batch_size=16,
)

print(validation_result["metrics"])
pd.DataFrame(validation_result["predictions"])


{'confusion_matrix': [[114, 1, 2], [1, 140, 1], [0, 3, 18]], 'accuracy': 0.9714285714285714, 'precision': [0.991304347826087, 0.9722222222222222, 0.8571428571428571], 'recall': [0.9743589743589743, 0.9859154929577465, 0.8571428571428571], 'class_f1': [0.9827586206896551, 0.979020979020979, 0.8571428571428571], 'macro_f1': 0.9396408189511637}


,filename,actual,predicted
0,100Cr6_AR_06_10.png,Brittle,Brittle
1,100Cr6_AR_06_3.png,Brittle,Brittle
2,304_H2_02_1.png,Brittle,Brittle
3,304_H2_02_10.png,Brittle,Brittle
4,304_H2_02_11.png,Brittle,Brittle
...,...,...,...
275,mX65_H2_29_22.png,Mixed,Mixed
276,mX65_H2_29_23.png,Mixed,Ductile
277,mX65_H2_29_3.png,Mixed,Mixed
278,mX65_H2_29_31.png,Mixed,Mixed


## Test

After selecting the checkpoint with validation, set `RUN_FINAL_TEST_EVALUATION = True` in the setup cell and run the cell below once. It uses the original `dataset_split.csv`; generated Mixed images belong to training only.


In [19]:
RUN_FINAL_TEST_EVALUATION = True

In [20]:
if RUN_FINAL_TEST_EVALUATION:
    final_test_result = run_prediction(
        data_dir=DATA_DIR,
        model_key=MODEL_KEY,
        checkpoint_path=CHECKPOINT_PATH,
        split="test",
        split_file="dataset_split_augmented.csv",
        device=DEVICE,
        batch_size=16,
    )
    print("Final held-out test metrics:", final_test_result["metrics"])
else:
    print("Final test evaluation is disabled.")


Final held-out test metrics: {'confusion_matrix': [[112, 0, 4], [1, 134, 8], [3, 6, 11]], 'accuracy': 0.921146953405018, 'precision': [0.9655172413793104, 0.9571428571428572, 0.4782608695652174], 'recall': [0.9655172413793104, 0.9370629370629371, 0.55], 'class_f1': [0.9655172413793104, 0.9469964664310955, 0.5116279069767442], 'macro_f1': 0.80804720492905}
